# Hanabi - Data Engineering

### Game data preparation

The data is obtained from https://github.com/yawgmoth/HanabiData.

In [133]:
import re
import random
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import zipfile
import io
import random

The repo is 9 years old and contains code made with Python 2. They included the following code to create the deck with a random seed given in the game logs:

In [134]:
# def make_deck(seed):
#     random.seed(seed)
#     deck = []
#     for col in ["green", "yellow", "white", "blue", "red"]:
#         for num, cnt in enumerate([3,2,2,2,1]):
#             for i in xrange(cnt):
#                 deck.append((col, num+1))
#     random.shuffle(deck)
#     return deck

We had to change the logic to Python 3 and used GenAI to create a function which simulates the Python 2 shuffle:

In [135]:
def py2_shuffle(items):
    for i in range(len(items) - 1, 0, -1):
        j = int(random.random() * (i + 1))
        items[i], items[j] = items[j], items[i]

def make_deck(seed):
    colors = ["green", "yellow", "white", "blue", "red"]
    random.seed(seed)
    deck = []
    for col in colors:
        for num, cnt in enumerate([3, 2, 2, 2, 1]):
            for i in range(cnt):
                deck.append((col, num + 1))
    
    py2_shuffle(deck)
    return deck

This function extracts the start cards for both players by using the make_deck function:

In [136]:
def process_hanabi_log(log_content, filename):
    lines = log_content.split('\n')

    seed = None
    final_score = None
    first_player = None

    for line in lines:
        line = line.strip()

        if line.startswith("Treatment:"): #after the treatment, we can find the seed for the deck
            match = re.search(r",\s*(\d+)\)", line)
            if match:
                seed = int(match.group(1)) #the seed used for the make_deck function

        elif line.startswith("MOVE:") and first_player is None: #defines the first player
            first_player = int(line.split()[1]) #0 is the AI, 1 is the human player

        elif line.startswith("Score"):
            final_score = int(line.split()[-1]) #the final score is found at the end of the line after "Score:"

    if seed is None or first_player is None:
        return None  # Incomplete file skipped

    deck = make_deck(seed)

    if first_player == 0: #0 is the AI
        p1_cards, p2_cards = deck[0:5], deck[5:10] # p1 is the AI
    else: #1 is the human player
        p2_cards, p1_cards = deck[0:5], deck[5:10] # p2 is the human player

  #one hot encoding
    colors = ["green", "yellow", "white", "blue", "red"]
    row_data = {"file_source": filename, "final_score": final_score}
    
    for player_prefix, hand in [("p1", p1_cards), ("p2", p2_cards)]:
        for col in colors:
            for num in range(1, 6):
                col_name = f"{player_prefix}_{col}{num}"
                row_data[col_name] = hand.count((col, num)) #counts how many times the card appears
                
    return row_data

Download the data from the GitHub Repo.

In [137]:
def download_repo_zip():
    """loading the whole repo as zip to the RAM (1 Request)."""
    url = "https://github.com/yawgmoth/HanabiData/archive/refs/heads/master.zip"
    print("Loading repository as ZIP...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    print(f"Download finished ({len(response.content) / 1_000_000:.1f} MB).")
    return zipfile.ZipFile(io.BytesIO(response.content))


def build_dataset(output_csv="hanabi_dataset_raw.csv"):
    """Process all .log files and save the result as a CSV."""
    zf = download_repo_zip()

    # Filter all .log files in the log/ directory
    log_files = [
        f for f in zf.namelist()
        if "/log/" in f and f.endswith(".log")
    ]
    print(f"{len(log_files)} Log files found.\n")

    rows = []
    errors = []

    for i, filepath in enumerate(log_files):
        filename = filepath.split("/")[-1]
        try:
            with zf.open(filepath) as f:
                content = f.read().decode("utf-8")
            row = process_hanabi_log(content, filename)
            if row is not None:
                rows.append(row)
        except Exception as e:
            errors.append((filename, str(e)))

        # Progression
        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/{len(log_files)} processed...")

    print(f"\nDone: {len(rows)} rows extracted, {len(errors)} Errors.")

    if errors:
        print("Corrupted files:")
        for name, err in errors[:10]:  # show a maximum of 10
            print(f"  {name}: {err}")

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"\nDataset saved as: {output_csv}")
    print(df.head())
    return df

Create the csv-file.

In [138]:
df = build_dataset()

Loading repository as ZIP...
Download finished (2.6 MB).
2280 Log files found.

  200/2280 processed...
  400/2280 processed...
  600/2280 processed...
  800/2280 processed...
  1000/2280 processed...
  1200/2280 processed...
  1400/2280 processed...
  1600/2280 processed...
  1800/2280 processed...
  2000/2280 processed...
  2200/2280 processed...

Done: 2040 rows extracted, 0 Errors.

Dataset saved as: hanabi_dataset_raw.csv
                file_source  final_score  p1_green1  p1_green2  p1_green3  \
0  game003d9bcb9d27dacf.log           15          1          0          0   
1  game0073425f0b25520f.log           17          1          0          0   
2  game007c0324e3f7b88a.log           19          0          0          0   
3  game007e20cb959b48fa.log           12          1          1          0   
4  game00abb0a354af727f.log            9          0          1          1   

   p1_green4  p1_green5  p1_yellow1  p1_yellow2  p1_yellow3  ...  p2_blue1  \
0          0          0     

Check the first few rows of the dataset.

In [139]:
df.head

<bound method NDFrame.head of                    file_source  final_score  p1_green1  p1_green2  p1_green3  \
0     game003d9bcb9d27dacf.log           15          1          0          0   
1     game0073425f0b25520f.log           17          1          0          0   
2     game007c0324e3f7b88a.log           19          0          0          0   
3     game007e20cb959b48fa.log           12          1          1          0   
4     game00abb0a354af727f.log            9          0          1          1   
...                        ...          ...        ...        ...        ...   
2035  gameff7fdd9b2cd0e110.log           16          1          1          0   
2036  gameff840d8a63adea1d.log           17          0          0          0   
2037  gameffd76bbcfdea1d43.log            2          0          0          0   
2038  gameffd8a16ab95eee2f.log            6          1          0          0   
2039  gameffe555cd8f743e34.log           15          0          1          1   

      p1_

### Feature engineering

In [140]:
colors = ["green", "yellow", "white", "blue", "red"]

for player_prefix in ["p1", "p2"]:
    for num in range(1, 6):
        cols = [f"{player_prefix}_{col}{num}" for col in colors]
        df[f"{player_prefix}_{num}_count"] = df[cols].sum(axis=1)
    
    df[f"{player_prefix}_hand_sum"] = sum(
        df[f"{player_prefix}_{num}_count"] * num for num in range(1, 6)
    )
    df[f"{player_prefix}_avg_value"] = df[f"{player_prefix}_hand_sum"] / 5

df.head()
print(df.columns.values)
df


<ArrowStringArray>
[ 'file_source',  'final_score',    'p1_green1',    'p1_green2',
    'p1_green3',    'p1_green4',    'p1_green5',   'p1_yellow1',
   'p1_yellow2',   'p1_yellow3',   'p1_yellow4',   'p1_yellow5',
    'p1_white1',    'p1_white2',    'p1_white3',    'p1_white4',
    'p1_white5',     'p1_blue1',     'p1_blue2',     'p1_blue3',
     'p1_blue4',     'p1_blue5',      'p1_red1',      'p1_red2',
      'p1_red3',      'p1_red4',      'p1_red5',    'p2_green1',
    'p2_green2',    'p2_green3',    'p2_green4',    'p2_green5',
   'p2_yellow1',   'p2_yellow2',   'p2_yellow3',   'p2_yellow4',
   'p2_yellow5',    'p2_white1',    'p2_white2',    'p2_white3',
    'p2_white4',    'p2_white5',     'p2_blue1',     'p2_blue2',
     'p2_blue3',     'p2_blue4',     'p2_blue5',      'p2_red1',
      'p2_red2',      'p2_red3',      'p2_red4',      'p2_red5',
   'p1_1_count',   'p1_2_count',   'p1_3_count',   'p1_4_count',
   'p1_5_count',  'p1_hand_sum', 'p1_avg_value',   'p2_1_count',
   'p2

,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,...,p1_5_count,p1_hand_sum,p1_avg_value,p2_1_count,p2_2_count,p2_3_count,p2_4_count,p2_5_count,p2_hand_sum,p2_avg_value
0,game003d9bcb9d27dacf.log,15,1,0,0,0,0,1,1,0,...,1,10,2.0,2,1,0,2,0,12,2.4
1,game0073425f0b25520f.log,17,1,0,0,0,0,0,0,0,...,0,16,3.2,3,2,0,0,0,7,1.4
2,game007c0324e3f7b88a.log,19,0,0,0,2,0,1,0,0,...,0,12,2.4,1,2,1,1,0,12,2.4
3,game007e20cb959b48fa.log,12,1,1,0,0,0,0,0,0,...,1,15,3.0,2,0,1,2,0,13,2.6
4,game00abb0a354af727f.log,9,0,1,1,0,0,0,0,0,...,1,16,3.2,2,2,0,1,0,10,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2035,gameff7fdd9b2cd0e110.log,16,1,1,0,0,0,0,1,0,...,0,8,1.6,2,0,2,0,1,13,2.6
2036,gameff840d8a63adea1d.log,17,0,0,0,0,0,2,1,0,...,0,6,1.2,1,3,0,0,1,12,2.4
2037,gameffd76bbcfdea1d43.log,2,0,0,0,1,0,0,1,0,...,0,17,3.4,2,1,2,0,0,10,2.0
2038,gameffd8a16ab95eee2f.log,6,1,0,0,0,1,1,0,0,...,1,12,2.4,0,3,1,0,1,14,2.8


### Merging & Clean up

As mentioned in 1_DatasetCharacteristics, some starting hands are overrepresented in the dataset. To remove this bias we decided to only pick two random games from the multiples and discard the rest.
Since there is no column to merge game logs with the games.csv. It should be done prior to that.

##### Merging

In [141]:
url = "https://raw.githubusercontent.com/yawgmoth/HanabiData/master/games.csv"
df_games = pd.read_csv(url)
df_games

,id,ai,deck,score,time,first
0,bcbbc9bc2cd7369f,intentional,3,18,1.490489e+09,yes
1,19dbddedf07994e2,full,9813,5,1.489966e+09,no
2,ca0922b34e670338,full,1965,20,1.491855e+09,no
3,cfae3ca2f65f0b94,full,1335,5,1.490366e+09,no
4,f87d30729e499d72,outer,9695,11,1.491411e+09,no
...,...,...,...,...,...,...
2035,ca0922b34e670338,outer,186,5,1.491853e+09,no
2036,19dbddedf07994e2,outer,6609,5,1.489739e+09,no
2037,f87d30729e499d72,outer,8455,14,1.491410e+09,no
2038,19dbddedf07994e2,full,1777,13,1.489829e+09,no


In [142]:

df_merged = pd.concat([df, df_games], axis=1)
print(len(df_merged))
df_merged

2040


,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,...,p2_4_count,p2_5_count,p2_hand_sum,p2_avg_value,id,ai,deck,score,time,first
0,game003d9bcb9d27dacf.log,15,1,0,0,0,0,1,1,0,...,2,0,12,2.4,bcbbc9bc2cd7369f,intentional,3,18,1.490489e+09,yes
1,game0073425f0b25520f.log,17,1,0,0,0,0,0,0,0,...,0,0,7,1.4,19dbddedf07994e2,full,9813,5,1.489966e+09,no
2,game007c0324e3f7b88a.log,19,0,0,0,2,0,1,0,0,...,1,0,12,2.4,ca0922b34e670338,full,1965,20,1.491855e+09,no
3,game007e20cb959b48fa.log,12,1,1,0,0,0,0,0,0,...,2,0,13,2.6,cfae3ca2f65f0b94,full,1335,5,1.490366e+09,no
4,game00abb0a354af727f.log,9,0,1,1,0,0,0,0,0,...,1,0,10,2.0,f87d30729e499d72,outer,9695,11,1.491411e+09,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2035,gameff7fdd9b2cd0e110.log,16,1,1,0,0,0,0,1,0,...,0,1,13,2.6,ca0922b34e670338,outer,186,5,1.491853e+09,no
2036,gameff840d8a63adea1d.log,17,0,0,0,0,0,2,1,0,...,0,1,12,2.4,19dbddedf07994e2,outer,6609,5,1.489739e+09,no
2037,gameffd76bbcfdea1d43.log,2,0,0,0,1,0,0,1,0,...,0,0,10,2.0,f87d30729e499d72,outer,8455,14,1.491410e+09,no
2038,gameffd8a16ab95eee2f.log,6,1,0,0,0,1,1,0,0,...,0,1,14,2.8,19dbddedf07994e2,full,1777,13,1.489829e+09,no


##### Clean up

In [143]:
hand_cols = [c for c in df.columns if c not in ["file_source", "final_score", "p1_hand_sum", "p2_hand_sum", "p1_total_cards", "p2_total_cards", "p1_avg_value", "p2_avg_value"]]

# if a starting hand is duplicated > 3 times, pick only 2 random
keep_indices = (
    df.groupby(hand_cols)
      .apply(
          lambda g: (
              g.sample(2, random_state=42).index
              if len(g) > 3
              else g.index
          )
      )
      .explode()
      .astype(int)
)

df_clean = df.loc[keep_indices].reset_index(drop=True)

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Original columns:", df.shape[1])
print("Cleaned columns:", df_clean.shape[1])

# Plot the distribution of starting hands in the clean dataset
state_frequencies = (
    df_clean[hand_cols]
    .value_counts()
    .sort_values(ascending=False)
)

# Convert each state tuple to a readable label
labels = []
for state in state_frequencies.index:
    active_features = [
        col
        for col, value in zip(hand_cols, state)
        if value != 0
    ]

    labels.append(", ".join(active_features) if active_features else "None")

Original rows: 2040
Cleaned rows: 1810
Original columns: 66
Cleaned columns: 66


Only keep the cleaned rows.

In [148]:
df_merged = df_merged[df_merged["file_source"].isin(df_clean["file_source"])]
print("Cleaned rows in df_merged:", len(df_merged))

Cleaned rows in df_merged: 1810


### Load in the additional info

Load the additional participants data and look at it.

In [152]:
url = "https://raw.githubusercontent.com/yawgmoth/HanabiData/master/participants.csv"
df_participants = pd.read_csv(url)
print("Number of unique participants in df_participants:", len(df_participants["id"].unique()))
print("Number of unique participants in df_merged:", len(df_merged["id"].unique()))
df_participants


Number of unique participants in df_participants: 240
Number of unique participants in df_merged: 219


,id,ai,deck,score,age,boardgameexp,gamer,hanabiexp,recent,maxscore,intention,skill,like,publish
0,b1ac9e5c7a6975d2,outer,3,4,30s,4,yes,4,4,4,3,2,2,yes
1,bcbbc9bc2cd7369f,intentional,3,18,30s,4,yes,4,4,4,2,2,2,yes
2,a992a1fd533fcd43,full,4,9,30s,4,yes,4,4,3,5,1,2,yes
3,53d46d46c9ef0e14,intentional,4,13,20s,4,yes,2,1,1,3,2,1,yes
4,1b17ab93ed1184b4,full,3,8,40s,4,yes,2,2,1,4,4,3,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,4129b562fe425a0b,full,4,2,20s,4,yes,4,4,4,4,1,4,yes
236,e94bf9e6daf555e4,intentional,1,14,,,,,,,,,,yes
237,591767f1643bd690,full,1,0,,,,,,,,2,,yes
238,767257f0a1f98ea8,outer,4,18,20s,4,yes,2,2,1,4,3,4,yes


Merge it with the gamelogs data.

In [2]:

df_merged_final = pd.merge(df_merged, df_participants, on="id", how="left")
print("Number of rows in df_merged_final:", len(df_merged_final))
df_merged_final

NameError: name 'pd' is not defined

In [1]:
# save the clean dataset as a csv file
df_merged_final.to_csv("hanabi_dataset_clean_merged.csv", index=False)

NameError: name 'df_merged_final' is not defined

Some games do not have more data on the participants. How many?

In [160]:
print(df_merged_final["age"].isna().sum())  # NaN?
print((df_merged_final["age"] == "").sum()) # empty strings?
print((df_merged_final["age"].str.strip() == "").sum())

107
0
0


Replace them with NaN to better work with them.

In [162]:
df_merged_final["age"] = df_merged_final["age"].str.strip().replace("", np.nan)
df_merged_final.head()

,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,...,age,boardgameexp,gamer,hanabiexp,recent,maxscore,intention,skill,like,publish
0,game003d9bcb9d27dacf.log,15,1,0,0,0,0,1,1,0,...,30s,4,yes,4,4,4,2,2,2,yes
1,game0073425f0b25520f.log,17,1,0,0,0,0,0,0,0,...,30s,4,yes,4,4,4,5,3,3,yes
2,game007c0324e3f7b88a.log,19,0,0,0,2,0,1,0,0,...,40s,4,yes,3,4,2,3,3,3,yes
3,game007e20cb959b48fa.log,12,1,1,0,0,0,0,0,0,...,NaN,,,,,,,,,yes
4,game00dcc5b032e51393.log,16,0,0,1,1,0,0,1,0,...,40s,4,yes,3,4,2,3,3,3,yes
